# WasteVision Sorter: entrenamiento de la segmentación de instancias

Este cuaderno entrena la red que usa el simulador para ver los residuos sobre la cinta.

1. Genera el conjunto de datos sintético (recortes reales compuestos sobre la cinta, con máscaras exactas).
2. Entrena YOLO11-seg (variantes `s` y `n`) con GPU.
3. Evalúa en el conjunto de prueba, hecho con objetos que la red nunca vio (los del simulador).
4. Exporta a ONNX (480 x 640) para ejecutarlo en el navegador.

**Antes de empezar:** menú *Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4*. Luego *Ejecutar todo*. Tarda entre 60 y 90 minutos.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "Activa la GPU: Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4"

In [ ]:
# Opción A: el proyecto ya está en tu GitHub (deja USE_ZIP = False)
# Opción B: aún no hay repo. Pon USE_ZIP = True, ejecuta y sube wastevision-sorter.zip cuando lo pida.
USE_ZIP = True
REPO = "https://github.com/andresjsaco-sudo/wastevision-sorter.git"

import os, zipfile
os.chdir("/content")
!rm -rf /content/wastevision-sorter
if USE_ZIP:
    from google.colab import files
    up = files.upload()                      # selecciona wastevision-sorter.zip
    zipfile.ZipFile(next(iter(up))).extractall("/content")
else:
    !git clone --depth 1 $REPO
os.chdir("/content/wastevision-sorter")
!pip -q install ultralytics onnx onnxslim onnxruntime
!ls sprites


## 1. Conjunto de datos sintético

In [ ]:
!python training/make_dataset.py --out /content/dataset --train 4000 --val 400 --test 400

In [ ]:
# Muestra de control: imagen con las máscaras que se usarán como etiquetas
import cv2, glob, random, numpy as np, matplotlib.pyplot as plt
names = ["plastic", "glass", "cardboard", "metal", "paper", "organic", "other"]
cols = [(242,193,78),(63,193,176),(194,143,92),(157,180,208),(236,231,214),(124,207,95),(138,143,153)]
fig, axs = plt.subplots(2, 3, figsize=(18, 9))
for ax, f in zip(axs.ravel(), random.sample(sorted(glob.glob("/content/dataset/images/train/*.jpg")), 6)):
    im = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB); ov = im.copy()
    for line in open(f.replace("images", "labels").replace(".jpg", ".txt")):
        v = line.split(); c = int(v[0]); p = (np.array(v[1:], float).reshape(-1, 2) * [640, 480]).astype(np.int32)
        cv2.fillPoly(ov, [p], cols[c]); cv2.polylines(im, [p], True, cols[c], 2)
    ax.imshow(cv2.addWeighted(ov, 0.45, im, 0.55, 0)); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Entrenamiento

La escala y el punto de vista están fijos en la simulación (cámara cenital, 2,5 mm por píxel), así que la aumentación se limita a lo que sí varía: volteos, color, exposición y un poco de mosaico.

In [ ]:
from ultralytics import YOLO
DATA = "/content/dataset/data.yaml"
MODELS = ["yolo11s-seg", "yolo11n-seg"]   # deja solo uno si tienes poco tiempo
EPOCHS = 60

def train(name):
    YOLO(f"{name}.pt").train(
        data=DATA, imgsz=640, epochs=EPOCHS, batch=16, patience=15, seed=0, workers=2,
        mosaic=0.3, close_mosaic=10, scale=0.15, translate=0.05, degrees=0.0, fliplr=0.5, flipud=0.5,
        hsv_h=0.01, hsv_s=0.4, hsv_v=0.3, erasing=0.0, copy_paste=0.0,
        project="/content/runs", name=name, exist_ok=True, plots=True)
    return YOLO(f"/content/runs/{name}/weights/best.pt")

trained = {name: train(name) for name in MODELS}

## 3. Evaluación en objetos nunca vistos

In [ ]:
import json, shutil, os
report = {}
for name, m in trained.items():
    r = m.val(data=DATA, split="test", imgsz=640, batch=16, project="/content/runs", name=f"{name}-test", exist_ok=True, plots=True)
    cm = r.confusion_matrix.matrix
    report[name] = {
        "box_map50": float(r.box.map50), "box_map50_95": float(r.box.map),
        "mask_map50": float(r.seg.map50), "mask_map50_95": float(r.seg.map),
        "mask_precision": float(r.seg.mp), "mask_recall": float(r.seg.mr),
        "per_class_mask_map50_95": {names[int(c)]: float(v) for c, v in zip(r.ap_class_index, r.seg.maps[r.ap_class_index])},
        "per_class_mask_map50": {names[int(c)]: float(v) for c, v in zip(r.ap_class_index, r.seg.ap50)},
        "confusion_matrix_rows_pred_cols_true": cm.tolist(),
        "gpu_ms_per_image": r.speed, "params": int(sum(p.numel() for p in m.model.parameters())),
    }
    print(name, {k: round(v, 4) for k, v in report[name].items() if isinstance(v, float)})

## 4. Exportar a ONNX para el navegador

In [ ]:
os.makedirs("web/models", exist_ok=True)
entries = []
for i, (name, m) in enumerate(trained.items()):
    onnx_path = m.export(format="onnx", imgsz=(480, 640), opset=12, simplify=True, nms=False, dynamic=False, half=False)
    dst = f"web/models/wastevision-{name}.onnx"; shutil.copy(onnx_path, dst)
    report[name]["onnx_mb"] = os.path.getsize(dst) / 1e6
    entries.append({"id": name, "name": name.replace("yolo", "YOLO"), "file": f"models/wastevision-{name}.onnx",
                    "imgsz": [480, 640], "available": True, "default": i == 0})
json.dump({"models": entries}, open("web/models/models.json", "w"), indent=2)
json.dump(report, open("web/models/report.json", "w"), indent=2)

# comprobación: el ONNX exportado responde con la forma que espera la página
import onnxruntime as ort, numpy as np
for e in entries:
    s = ort.InferenceSession("web/" + e["file"], providers=["CPUExecutionProvider"])
    o = s.run(None, {s.get_inputs()[0].name: np.zeros((1, 3, 480, 640), np.float32)})
    print(e["id"], [x.shape for x in o])   # se espera (1, 43, 6300) y (1, 32, 120, 160)

In [ ]:
# Paquete con los modelos, el informe y las gráficas de entrenamiento
!mkdir -p /content/pack/plots && cp web/models/* /content/pack/
for name in trained:
    for f in ["results.csv", "results.png", "confusion_matrix_normalized.png", "MaskPR_curve.png", "val_batch0_pred.jpg"]:
        for d in [name, f"{name}-test"]:
            src = f"/content/runs/{d}/{f}"
            if os.path.exists(src): shutil.copy(src, f"/content/pack/plots/{d}_{f}")
!cd /content/pack && zip -qr /content/wastevision_models.zip .
from google.colab import files; files.download("/content/wastevision_models.zip")

## 5. Qué hacer con el resultado

Se descarga `wastevision_models.zip`. Hay dos caminos:

- **Enviárselo a Claude** en la conversación: integra los modelos, publica la página y actualiza el artículo con las métricas.
- **Subirlo tú:** en GitHub, entra a `web/models/`, usa *Add file > Upload files* y sube los dos `.onnx`, `models.json` y `report.json`. Vercel publica solo.